# Personalized Recommendation Engine — Hybrid Collaborative + Content-Based Ranking System

Architecture
------------
1. CollaborativeFilteringModel
   - Builds a user-item interaction matrix (implicit feedback: views/purchases/ratings)
   - Uses Truncated SVD (a matrix-factorization technique, analogous to funkSVD/ALS)
     to learn latent user and item embeddings
   - Produces a "users who behaved like you also liked this" score

2. ContentBasedModel
   - Builds item embeddings from item metadata (title/category/description) via TF-IDF
   - Computes similarity between items a user already engaged with and candidate items
   - Produces a "this is similar to what you already liked" score

3. HybridRecommender
   - Normalizes both scores to [0, 1]
   - Combines them with a tunable weight (alpha) — this is standard industry practice
     (e.g. Netflix/Amazon blend collaborative + content signals, often with a learned
     weight rather than a fixed one; here we expose alpha as a simple hyperparameter)
   - Filters out items the user has already interacted with
   - Returns a ranked Top-N list

Why hybrid instead of either alone?
- Pure collaborative filtering suffers from the "cold start" problem: new items/users
  with no interaction history get no signal.
- Pure content-based filtering tends to over-specialize (recommends only near-duplicates
  of what a user already liked, no serendipity/discovery).
- Hybridizing gives cold-start coverage (content-based fallback) + collaborative signal
  quality (wisdom-of-the-crowd) when data is available.

In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [2]:
# ----------------------------------------------------------------------------
# 1. Synthetic data generation (stand-in for a real user-item interaction log
#    and item catalog — swap this out for your actual data source in production)
# ----------------------------------------------------------------------------
def generate_synthetic_data(n_users=500, n_items=200, n_interactions=6000, seed=42):
    rng = np.random.default_rng(seed)

    categories = ["Electronics", "Books", "Fashion", "Home", "Sports", "Beauty", "Toys"]
    item_ids = np.arange(n_items)
    item_category = rng.choice(categories, size=n_items)
    item_titles = [
        f"{cat} Product {i} — premium {cat.lower()} item with great reviews"
        for i, cat in zip(item_ids, item_category)
    ]
    items = pd.DataFrame({
        "item_id": item_ids,
        "category": item_category,
        "description": item_titles,
    })

    # Simulate users with a latent preference for 1-2 categories (creates real structure
    # for collaborative filtering to discover, rather than pure noise)
    user_pref_category = rng.choice(categories, size=n_users)

    user_ids = rng.integers(0, n_users, size=n_interactions)
    interactions = []
    for u in user_ids:
        pref_cat = user_pref_category[u]
        # 70% chance of interacting within their preferred category, 30% exploring
        if rng.random() < 0.7:
            candidate_items = items[items.category == pref_cat].item_id.values
        else:
            candidate_items = items.item_id.values
        item = rng.choice(candidate_items)
        rating = rng.integers(1, 6)  # implicit "strength" signal 1-5
        interactions.append((u, item, rating))

    interactions_df = pd.DataFrame(interactions, columns=["user_id", "item_id", "rating"])
    interactions_df = interactions_df.groupby(["user_id", "item_id"], as_index=False).rating.max()
    return interactions_df, items

In [3]:
# ----------------------------------------------------------------------------
# 2. Collaborative Filtering (matrix factorization)
# ----------------------------------------------------------------------------
class CollaborativeFilteringModel:
    def __init__(self, n_factors=20, seed=42):
        self.n_factors = n_factors
        self.seed = seed

    def fit(self, interactions_df, n_users, n_items):
        self.n_users, self.n_items = n_users, n_items
        matrix = np.zeros((n_users, n_items))
        for _, row in interactions_df.iterrows():
            matrix[int(row.user_id), int(row.item_id)] = row.rating
        self.raw_matrix = matrix

        self.svd = TruncatedSVD(n_components=self.n_factors, random_state=self.seed)
        self.user_factors = self.svd.fit_transform(matrix)          # (n_users, k)
        self.item_factors = self.svd.components_.T                   # (n_items, k)
        return self

    def score(self, user_id):
        """Predicted affinity score for every item for a given user."""
        return self.user_factors[user_id] @ self.item_factors.T

In [4]:
# ----------------------------------------------------------------------------
# 3. Content-Based Filtering (TF-IDF item similarity)
# ----------------------------------------------------------------------------
class ContentBasedModel:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words="english")

    def fit(self, items_df):
        self.items_df = items_df.reset_index(drop=True)
        self.item_vectors = self.vectorizer.fit_transform(items_df.description)
        self.similarity_matrix = cosine_similarity(self.item_vectors)
        return self

    def score(self, user_item_ids):
        """
        Average similarity of every catalog item to the set of items the user
        already engaged with — items similar to a user's history score higher.
        """
        if len(user_item_ids) == 0:
            return np.zeros(self.similarity_matrix.shape[0])
        return self.similarity_matrix[user_item_ids].mean(axis=0)

In [5]:
# ----------------------------------------------------------------------------
# 4. Hybrid Recommender — blends both signals and produces final ranking
# ----------------------------------------------------------------------------
class HybridRecommender:
    def __init__(self, alpha=0.6, n_factors=20):
        """
        alpha: weight on the collaborative-filtering score vs content-based score.
               alpha=1.0 -> pure collaborative, alpha=0.0 -> pure content-based.
               0.6 is a common production starting point, then A/B tested.
        """
        self.alpha = alpha
        self.cf_model = CollaborativeFilteringModel(n_factors=n_factors)
        self.cb_model = ContentBasedModel()
        self.scaler = MinMaxScaler()

    def fit(self, interactions_df, items_df, n_users, n_items):
        self.interactions_df = interactions_df
        self.items_df = items_df
        self.cf_model.fit(interactions_df, n_users, n_items)
        self.cb_model.fit(items_df)
        return self

    def recommend(self, user_id, top_n=10):
        user_history = self.interactions_df[self.interactions_df.user_id == user_id].item_id.values

        cf_scores = self.cf_model.score(user_id)
        cb_scores = self.cb_model.score(user_history)

        # Normalize both score vectors to [0, 1] so alpha blending is meaningful
        cf_norm = self.scaler.fit_transform(cf_scores.reshape(-1, 1)).flatten()
        cb_norm = self.scaler.fit_transform(cb_scores.reshape(-1, 1)).flatten()

        hybrid_scores = self.alpha * cf_norm + (1 - self.alpha) * cb_norm

        # Filter out items already seen
        hybrid_scores[user_history] = -np.inf

        top_idx = np.argsort(-hybrid_scores)[:top_n]
        recs = self.items_df.iloc[top_idx].copy()
        recs["score"] = hybrid_scores[top_idx]
        return recs.reset_index(drop=True)

In [7]:
# ----------------------------------------------------------------------------
# 5. Evaluation — Precision@K on a held-out interaction per user
# ----------------------------------------------------------------------------
def evaluate_precision_at_k(model, interactions_df, k=10, n_eval_users=100, seed=42):
    rng = np.random.default_rng(seed)
    eval_users = rng.choice(interactions_df.user_id.unique(), size=n_eval_users, replace=False)
    hits = 0
    for u in eval_users:
        user_items = interactions_df[interactions_df.user_id == u].item_id.values
        if len(user_items) < 2:
            continue
        held_out = rng.choice(user_items)
        # temporarily remove held-out item from history to test if model recovers it
        train_interactions = interactions_df[
            ~((interactions_df.user_id == u) & (interactions_df.item_id == held_out))
        ]
        model.interactions_df = train_interactions
        recs = model.recommend(u, top_n=k)
        if held_out in recs.item_id.values:
            hits += 1
    model.interactions_df = interactions_df  # restore
    return hits / n_eval_users

if __name__ == "__main__":
    N_USERS, N_ITEMS = 500, 200
    interactions_df, items_df = generate_synthetic_data(n_users=N_USERS, n_items=N_ITEMS)

    model = HybridRecommender(alpha=0.6, n_factors=20)
    model.fit(interactions_df, items_df, N_USERS, N_ITEMS)

    print("=== Top-10 Hybrid Recommendations for User 0 ===")
    print(model.recommend(user_id=0, top_n=10)[["item_id", "category", "score"]])

    precision = evaluate_precision_at_k(model, interactions_df, k=10, n_eval_users=100)
    print(f"\nPrecision@10 (held-out item recovery): {precision:.3f}")

=== Top-10 Hybrid Recommendations for User 0 ===
   item_id category     score
0       26  Fashion  0.746703
1       76  Fashion  0.720685
2       65  Fashion  0.693717
3       95  Fashion  0.688345
4       67  Fashion  0.687611
5       21  Fashion  0.664963
6      153  Fashion  0.659307
7      166  Fashion  0.637009
8      143  Fashion  0.626809
9      162  Fashion  0.624240

Precision@10 (held-out item recovery): 0.580
